In [ ]:
myproject/
├── manage.py
├── myproject/
│   ├── __init__.py
│   ├── settings.py
│   ├── urls.py
│   ├── asgi.py
│   └── wsgi.py
├── category/
│   ├── __init__.py
│   ├── admin.py
│   ├── apps.py
│   ├── migrations/
│   │   └── __init__.py
│   ├── models.py
│   ├── serializers.py
│   ├── views.py
│   ├── urls.py
│   ├── retry_service.py
│   ├── exceptions.py
│   ├── middleware.py
│   └── logging.conf
└── requirements.txt
```


In [ ]:
INSTALLED_APPS = [
    'django.contrib.admin',
    'django.contrib.auth',
    'django.contrib.contenttypes',
    'django.contrib.sessions',
    'django.contrib.messages',
    'django.contrib.staticfiles',
    'rest_framework',
    'corsheaders',
    'category',
]

MIDDLEWARE = [
    'django.middleware.security.SecurityMiddleware',
    'django.contrib.sessions.middleware.SessionMiddleware',
    'corsheaders.middleware.CorsMiddleware',
    'django.middleware.common.CommonMiddleware',
    'django.middleware.csrf.CsrfViewMiddleware',
    'django.contrib.auth.middleware.AuthenticationMiddleware',
    'django.contrib.messages.middleware.MessageMiddleware',
    'django.middleware.clickjacking.XFrameOptionsMiddleware',
]

CORS_ALLOW_ALL_ORIGINS = True
```

---



In [ ]:
from django.db import models

class Category(models.Model):
    name = models.CharField(max_length=100, unique=True)
    description = models.TextField(blank=True)
    created_at = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return self.name


In [ ]:
# 🧩 3. serializers.py

from rest_framework import serializers
from .models import Category

class CategorySerializer(serializers.ModelSerializer):
    class Meta:
        model = Category
        fields = '__all__'



In [ ]:

# 🧩 4. retry_service.py

```python
import requests
from tenacity import retry, stop_after_attempt, wait_fixed

class RetryService:

    @retry(stop=stop_after_attempt(3), wait=wait_fixed(2))
    def call_api(self, url):
        response = requests.get(url, timeout=5)
        if response.status_code != 200:
            raise Exception("External API Failed")
        return response.json()
```


In [ ]:

# 🧩 5. views.py

```python
from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework import status
from .models import Category
from .serializers import CategorySerializer
from .retry_service import RetryService
import logging

logger = logging.getLogger(__name__)

class CategoryListCreate(APIView):

    def get(self, request):
        categories = Category.objects.all()
        serializer = CategorySerializer(categories, many=True)
        return Response(serializer.data)

    def post(self, request):
        retry = RetryService()
        try:
            retry.call_api("https://httpbin.org/status/500")
        except Exception as e:
            logger.error(f"Retry failed: {e}")
            return Response({"error": "External API failed after retries"}, status=503)

        serializer = CategorySerializer(data=request.data)
        if serializer.is_valid():
            serializer.save()
            return Response(serializer.data, status=201)
        return Response(serializer.errors, status=400)


In [ ]:


class CategoryDetail(APIView):

    def get_object(self, pk):
        try:
            return Category.objects.get(pk=pk)
        except Category.DoesNotExist:
            return None

    def get(self, request, pk):
        category = self.get_object(pk)
        if not category:
            return Response({"error": "Not found"}, status=404)
        serializer = CategorySerializer(category)
        return Response(serializer.data)

    def put(self, request, pk):
        category = self.get_object(pk)
        if not category:
            return Response({"error": "Not found"}, status=404)

        serializer = CategorySerializer(category, data=request.data)
        if serializer.is_valid():
            serializer.save()
            return Response(serializer.data)
        return Response(serializer.errors, status=400)

    def delete(self, request, pk):
        category = self.get_object(pk)
        if not category:
            return Response({"error": "Not found"}, status=404)
        category.delete()
        return Response(status=204)
```

---


In [ ]:

# 🧩 6. urls.py (inside category app)

```python
from django.urls import path
from .views import CategoryListCreate, CategoryDetail

urlpatterns = [
    path('', CategoryListCreate.as_view()),
    path('<int:pk>/', CategoryDetail.as_view()),
]
```

---

# 🧩 7. main urls.py

```python
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('api/category/', include('category.urls')),
]
```

---

# 🧩 8. requirements.txt

```
django
djangorestframework
django-cors-headers
tenacity
requests
```

---

If you want I can also add:
✅ Swagger Docs
✅ Token/JWT Authentication
✅ Pagination & Filtering
✅ Docker Support

Just tell me! 🚀